## Data Extraction, Chunk Cleaning and Preparation

In [1]:
from bs4 import BeautifulSoup
import requests

response = requests.get(r"https://www.consumer.vic.gov.au/housing/renting/rent-bond-bills-and-condition-reports/paying-for-utilities-phone-and-internet")


In [2]:
html = response.content.decode("utf-8")

html

'\r\n<!DOCTYPE HTML>\r\n\r\n<html class="no-js" lang="en">\r\n\r\n  <head><meta charset="utf-8" /><meta http-equiv="X-UA-Compatible" content="IE=edge" />\r\n\r\n    <!-- Google Tag Manager Data Layer -->\r\n    <script>\r\n        window.dataLayer = [\r\n        {\r\n//          \'countryCode\' : \'{{Country Code}}\', // AU\r\n            \'language\' : \'\', // English\r\n            \'environment\' : \'Production\',\r\n            \'currentVersion\' : \'1.0.0.0\', // 1.0\r\n            \'pageType\' : \'Content Page\', // Home / Product / Category etc\r\n//          \'uiExperience\' : \'{{User Interface}}\', // Desktop / Mobile\r\n            \'user\' : {\r\n                \'id\' : \'\', // 0\r\n                \'type\' : \'Guest\' // Guest\r\n            },\r\n//          \'word-count-range\': {{Word Range}}, \r\n//          \'word-count\': {{Words}},\r\n            \'session-start\': \'1795562537\',\r\n            \'emergency\': document.querySelector(\'aside.notification-bar\') !=

In [3]:
with open(r"..\\raw_data\\Paying_for_utilities_and_services.txt", "w", encoding="utf-8") as f:
    f.write(html)

In [4]:
with open(r"..\\raw_data\\Paying_for_utilities_and_services.txt", "r", encoding="utf-8") as f:
    html = f.read()

In [5]:
from langchain_text_splitters import HTMLHeaderTextSplitter

headers_to_split_on = [("h1", "h1"), ("h2", "h2"), ("h3", "h3"), ("h4", "h4"), ("h5", "h5"), ("h6", "h6")]


html_header_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    return_each_element=False,
)

header_chunks = html_header_splitter.split_text(html)

In [6]:
header_chunks

[Document(metadata={}, page_content='Google Tag Manager (noscript) End Google Tag Manager (noscript)  \nTether, then Bootstrap JS. Prompt IE 6 users to install Chrome Frame. Remove this if you want to support IE 6.\n\n        chromium.org/developers/how-tos/chrome-frame-getting-started [if lt IE 7 ]>\n\n    <script src="//ajax.googleapis.com/ajax/libs/chrome-frame/1.0.3/CFInstall.min.js"></script>\n\n    <script>window.attachEvent(\'onload\',function(){CFInstall.check({mode:\'overlay\'})})</script>\n\n    <![endif]  \nSkip to content  \nSkip to main navigation  \nSkip to footer  \nBack  \nYour rights and responsibilities  \nMenu options for Consumer Affairs Victoria  \nHousing  \nRenting, buying and selling property, building and renovating, owners corporations, retirement villages  \nConsumers and businesses  \nBuying, selling and hiring products, services and cars, energy and utilities, product safety, scams, debt collectors, resources for businesses  \nLicensing and registration  \n

In [7]:
from langchain_text_splitters import HTMLSemanticPreservingSplitter

def table_to_text(table):
    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
    lines = []
    for tr in table.find_all("tr"):
        cells = [td.get_text(" ", strip=True).replace("\xa0", " ")
                 for td in tr.find_all("td")]
        if cells:
            lines.append("; ".join(f"{h}: {c}" for h, c in zip(headers, cells)))
    return "\n".join(lines)

html_semantic_splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=headers_to_split_on,
    max_chunk_size=1500,
    chunk_overlap=0,
    separators=["\n\n", "\n", ".", " "],
    elements_to_preserve=["table", "ul", "ol"],
    custom_handlers={"table": table_to_text},
    external_metadata= {"url": "the_url"}, # We add the url as we will implement RAG with Sources
    preserve_links=False,
)

semantic_chunks = html_semantic_splitter.split_text(html)

C:\Users\udhay\AppData\Local\Temp\ipykernel_95236\2036446212.py:13: LangChainBetaWarning: The class `HTMLSemanticPreservingSplitter` is in beta. It is actively being worked on, so the API may change.
  html_semantic_splitter = HTMLSemanticPreservingSplitter(


In [8]:
semantic_chunks

[Document(metadata={'url': 'the_url'}, page_content='window.dataLayer = [ { // \'countryCode\' : \'{{Country Code}}\', // AU \'language\' : \'\', // English \'environment\' : \'Production\', \'currentVersion\' : \'1.0.0.0\', // 1.0 \'pageType\' : \'Content Page\', // Home / Product / Category etc // \'uiExperience\' : \'{{User Interface}}\', // Desktop / Mobile \'user\' : { \'id\' : \'\', // 0 \'type\' : \'Guest\' // Guest }, // \'word-count-range\': {{Word Range}}, // \'word-count\': {{Words}}, \'session-start\': \'1795562537\', \'emergency\': document.querySelector(\'aside.notification-bar\') != null, }]; (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({\'gtm.start\': new Date().getTime(),event:\'gtm.js\'});var f=d.getElementsByTagName(s)[0], j=d.createElement(s),dl=l!=\'dataLayer\'?\'&l=\'+l:\'\';j.async=true;j.src= \'https://www.googletagmanager.com/gtm.js?id=\'+i+dl;f.parentNode.insertBefore(j,f); })(window,document,\'script\',\'dataLayer\',\'GTM-5BWZBTR\'); Paying for utilities and 

The structure of the web pages is such that we can advantage of it.

From the structure of the web pages on the website, we can also glean that all contents before the first `<h1>` header is mostly unrelated content. We can clean the chunks even more by eliminating the 'Sharethis page' chunk, which is a common noisy chunk in all the web pages.

All elements in the footer (starting with the chunk 'About us') is mostly noise.

Hence we can do some post-processing to eliminate such chunks.

In [9]:
import re

HEADER_KEY = re.compile(r"^h[1-6]$")

def clean_chunks(semantic_chunks):
    new_chunks = []
    header_1_met = False
    share_chunk_met = False
    footer_reached = False

    for i in range(0, len(semantic_chunks)):
        chunk = semantic_chunks[i]
        share_this_page_chunk = False
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                    if not header_1_met and HEADER_KEY.match(key):
                            if key == "h1":
                                header_1_met = True
                                break

                    if not share_chunk_met and value == "Sharethis page":
                            share_chunk_met = True
                            new_chunks[-1].page_content = chunk.page_content[20:].strip()
                            share_this_page_chunk = True
                            break

                    if key == "h3" and value == "About us":
                          footer_reached = True
                          break

        # If we reached the footer, there is no need to continue
        if footer_reached:
              break

        # Skip the chunk if we have not encountered our first header yet
        if not header_1_met or share_this_page_chunk:
             continue

        # Remove meaningless encodings from the chunk page_content
        chunk.page_content.replace("\xa0", " ")

        # Add the chunk
        new_chunks.append(chunk)

    return new_chunks

cleaned_semantic_chunks = clean_chunks(semantic_chunks)

In [10]:
cleaned_semantic_chunks

[Document(metadata={'h1': 'Paying for utilities and services', 'url': 'the_url'}, page_content='There are different rules for who pays for the ongoing cost of utilities and services, depending on what they are. setting up utilities and services You can also read about . On this page: Water and sewerage Electricity, gas and oil Phone and internet Television Very high usage charges Paying for utilities that do not meet efficiency standards Repayment for a utilities charge When a renter moves out Special rules for caravan parks replacing water, gas and electrical appliances Other pages have information about'),
 Document(metadata={'h2': 'Water and sewerage', 'url': 'the_url'}, page_content='If the property has its own meter, the renter must pay for the water they use, as well as the sewage disposal charge. This applies unless the rental provider (landlord) agrees to pay. If the rental provider agrees to pay, it should be in the rental agreement (lease) as a special condition. The rental p

We can also see that it is better to use the HTMLSemanticPreservingSplitter because of its 'custom_handlers' attribute that allows us to reformat table data and possibly other types of data in a more semantically meaningful way.

I also believe that HTMLSemanticPreservingSplitter will perform better than true Semantic Chunking because the structure of the web pages are very well-defined.

One big issue with HTMLSemanticPreservingSplitter is that each chunk has only its parent heading in its metadata, not all its ancestor heading. This can make our chunks lose meaningful context. In such case, the type of header metadata for headers provided by HTMLHeaderTextSplitter is highly preferred.

Direct metadata filtering on the headers is not reasonable as the header text is not exactly unique (meaning they could be written in multiple ways by the user, unlike product codes).

The best way to handle this is to preprocess the chunks so that the header tags are present as in HTMLHeaderTextSplitter and append all ancestor headers to the content as a prefix.

In [ ]:
import re

HEADER_KEY = re.compile(r"^h[1-6]$")

def post_process_header_tags(semantic_chunks):
    # List to contain header texts of h1-h6.
    headers = []
    # Variable to hold current smallest header traversed
    curr_smallest_header_traversed = 0
    for chunk in semantic_chunks:
        if chunk.metadata:
            for key, value in chunk.metadata.items():
                if HEADER_KEY.match(key):
                    header_index = int(key[-1])

                    # Adjust the header hierarchy for the chunk
                    if header_index == curr_smallest_header_traversed:
                        headers.pop()
                        headers.append((header_index,value))
                    elif header_index > curr_smallest_header_traversed:
                        curr_smallest_header_traversed = header_index
                        headers.append((header_index, value))
                    else:
                        curr_smallest_header_traversed = header_index
                        for i in range(1, len(headers) + 1):
                            if headers[-1][0] >= header_index:
                                headers.pop()
                            else:
                                break

                        headers.append((header_index, value))

                    chunk_header = ""

                    # Create the chunk header string
                    for header in headers:
                        chunk_header = chunk_header + header[1] + " - "

                    # Append all ancestor headers to the chunk content
                    chunk.page_content = chunk_header + chunk.page_content

In [12]:
post_process_header_tags(cleaned_semantic_chunks)


Headers: [(1, 'Paying for utilities and services')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Water and sewerage')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Electricity, gas and oil')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Electricity, gas and oil'), (3, 'Bottled gas')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Electricity, gas and oil'), (3, 'Metered electricity, gas or oil')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Electricity, gas and oil'), (3, 'Social housing heating and laundry costs')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Phone and internet')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Television')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Very high usage charges')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Paying for utilities that do not meet efficiency standards')]
Headers: [(1, 'Paying for utilities and services'), (2, 'Repayment 

In [13]:
cleaned_semantic_chunks

[Document(metadata={'h1': 'Paying for utilities and services', 'url': 'the_url'}, page_content='Paying for utilities and services - There are different rules for who pays for the ongoing cost of utilities and services, depending on what they are. setting up utilities and services You can also read about . On this page: Water and sewerage Electricity, gas and oil Phone and internet Television Very high usage charges Paying for utilities that do not meet efficiency standards Repayment for a utilities charge When a renter moves out Special rules for caravan parks replacing water, gas and electrical appliances Other pages have information about'),
 Document(metadata={'h2': 'Water and sewerage', 'url': 'the_url'}, page_content='Paying for utilities and services - Water and sewerage - If the property has its own meter, the renter must pay for the water they use, as well as the sewage disposal charge. This applies unless the rental provider (landlord) agrees to pay. If the rental provider agr

## Assesssing quality of retrieval with current chunking logic

Now, let us check how good vector similarity search works on our extracted chunks

In [14]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from load_dotenv import load_dotenv
import os

# Load our environment files into the environment
load_dotenv()

# We will use 'text-embedding-3-small' OpenAI model as both our prompt encoder and context encoder
encoder = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

if not os.path.exists("./chroma_db"):
    vector_store = Chroma.from_documents(
        documents=cleaned_semantic_chunks,
        embedding=encoder,
        persist_directory="./chroma_db"
    )
else:
    vector_store = Chroma(
        persist_directory="./chroma_db",
        embedding_function=encoder
    )

In [15]:
query = "What should a renter do when he moves out?"

In [16]:
results = vector_store.similarity_search(
    query=query,
    k=4
)

In [17]:
for i, doc in enumerate(results):
    print(f"=== Doc {i+1} ===")
    print(doc.page_content)
    print(doc.metadata)
    print("\n")

=== Doc 1 ===
Paying for utilities and services - When a renter moves out - When a renter moves out, they must tell the utilities companies (including water) the date they are leaving the property. This is so the companies know when to stop charging the renter and send them a final bill. Until a new renter moves in and the utilities companies have the new renter’s billing details, the rental provider must pay any utility costs after the previous renter moves out.
{'h2': 'When a renter moves out', 'url': 'the_url'}


=== Doc 2 ===
Paying for utilities and services - Repayment for a utilities charge - Notice to rental provider of rented premises (Word, 97KB) If the renter pays a charge that should have been paid by the rental provider, they can ask the rental provider to pay them back using a . The rental provider has 28 days to repay the charge. Notice to renter of rented premises (Word, 88KB) Similarly, if the rental provider pays a charge that the renter should have paid, they can ask

We can see that the retriever returned the most relevant content from the vector store as well as some noisy chunks. However, the most relevant content was indeed ranked as first.

Let us try a more complex query.

In [18]:
query = "I want to install a phone connection in my apartment. What steps should I take?"

In [19]:
results = vector_store.similarity_search(
    query=query,
    k=4
)

for i, doc in enumerate(results):
    print(f"=== Doc {i+1} ===")
    print(doc.page_content)
    print(doc.metadata)
    print("\n")

=== Doc 1 ===
Paying for utilities and services - Phone and internet - Renters must pay for the ongoing usage costs for phone and internet, unless the rental provider agrees otherwise. If the renter wants to set up phone and internet connections, they will have to ask the rental provider’s permission. There are two possible costs for connecting a property to a fixed telephone line (landline) for phone or internet use: a line connection fee, and a general connection fee. A general connection fee is charged when there is an existing phone line at the property but it is not currently connected. A line connection fee is what the telephone company will charge if a telephone line has not previously been set up at the property. This may be the case if the property is new or has been empty for some time (renters should check this before they sign the rental agreement). A renter who wants to set up a connection to an existing phone line at the property must pay the general connection fee. They 

Again, the results are highly satisfactory. The retriever returned the most relevant chunks with the first one being exactly what we need.

However, we have been testing with data from only 1 web page. Let us try fetching multiple pages and querying the relevant sections.

In [21]:
from glob import glob

web_files = glob("..\\raw_data\\*.txt")

web_files

['..\\raw_data\\Different_rental_agreements.txt',
 '..\\raw_data\\Finding_and_applying_for_a_rental_property.txt',
 '..\\raw_data\\Introduction_and_helpful_contacts.txt',
 '..\\raw_data\\Paying_for_utilities_and_services.txt',
 '..\\raw_data\\Residential_rental_agreements.txt']

In [22]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Create the encoder
encoder = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Create the new empty Chroma vector store
vector_store = Chroma(
    embedding_function=encoder,
    persist_directory="./chroma_db"
)

for file_path in web_files:

    with open(file_path, "r") as f:
        html = f.read()

    semantic_chunks = html_semantic_splitter.split_text(html)
    cleaned_semantic_chunks = clean_chunks(semantic_chunks)
    post_process_header_tags(cleaned_semantic_chunks)

    vector_store.add_documents(cleaned_semantic_chunks)

Headers: [(1, 'Different rental agreements')]
Headers: [(1, 'Different rental agreements'), (2, '')]
Headers: [(1, 'Different rental agreements'), (2, 'Feedback')]
Headers: [(1, 'Finding and applying for a rental property')]
Headers: [(1, 'Finding and applying for a rental property'), (3, '')]
Headers: [(1, 'Finding and applying for a rental property'), (3, 'Before you apply')]
Headers: [(1, 'Finding and applying for a rental property'), (3, 'Documents and information you may need to provide:')]
Headers: [(1, 'Finding and applying for a rental property'), (3, 'Unlawful discrimination in renting')]
Headers: [(1, 'Finding and applying for a rental property'), (3, 'Rules for advertised rental properties')]
Headers: [(1, 'Finding and applying for a rental property'), (3, 'Rules for advertised rental properties'), (4, 'Legal requirements:')]
Headers: [(1, 'Finding and applying for a rental property'), (3, 'Seen a rental property that doesn’t look quite right?')]
Headers: [(1, 'Finding and a

In [23]:
query = "What documents must I provide to my agent/renter when applying for a rental property?"

In [24]:
results = vector_store.similarity_search(
    query=query,
    k=4
)

for i, doc in enumerate(results):
    print(f"=== Doc {i+1} ===")
    print(doc.page_content)
    print(doc.metadata)
    print("\n")

=== Doc 1 ===
Finding and applying for a rental property - Documents and information you may need to provide: - Formal identification (such as a driver’s licence) Employment and financial details,  such as a payslip Rental or residential history Personal references, if requested. Rental providers must ask for this information through the prescribed rental application form. They can’t ask you for any information that is not set out in the form. applying for a property Read more about . The rental provider or agent cannot ask you for the following information in your application: Whether you’ve taken legal action or had a dispute with a previous rental provider. Your bond history, including any claims you’ve made on your bond. Detailed bank statements with daily transactions (if you need to provide a statement, you can delete transactions you think should be kept private).
{'url': 'the_url', 'h3': 'Documents and information you may need to provide:'}


=== Doc 2 ===
Residential rental ag

In [25]:
query = "I am new to Victoria. Who should I be calling to know more about the renting laws here?"

In [26]:
results = vector_store.similarity_search(
    query=query,
    k=4
)

for i, doc in enumerate(results):
    print(f"=== Doc {i+1} ===")
    print(doc.page_content)
    print(doc.metadata)
    print("\n")

=== Doc 1 ===
Introduction and helpful contacts - Introduction - Victoria has some of the strongest rental protections in Australia, but renting can still be complicated. With 1 in 3 Victorians now renting, understanding your rental rights has never been more important. Renters Guide The explains the rules and your rights at every stage of renting in Victoria, from applying for a property to moving out. Whether you’re a first-time renter or have rented before, this guide provides you with the information you need to rent confidently.
{'h3': 'Introduction', 'url': 'the_url'}


=== Doc 2 ===
Finding and applying for a rental property -  - Learn about finding and applying for a rental property in Victoria:
{'h3': '', 'url': 'the_url'}


=== Doc 3 ===
Introduction and helpful contacts - Stay updated - Renting For the most current information, go to . Facebook Instagram Follow Consumer Affairs Victoria on and for regular renting information and updates.
{'h3': 'Stay updated', 'url': 'the_ur

This one's result is not perfect. We might consider RAG optimisation techniques depending on our later recall rate. Let us try another one.

In [27]:
query = "My renter increased my rent without telling me. I believe this is not allowed. Who should I contact to clarify this?"

In [28]:
results = vector_store.similarity_search(
    query=query,
    k = 4
)

for i, doc in enumerate(results):
    print(f"=== Doc {i+1} ===")
    print(doc.page_content)
    print(doc.metadata)
    print("\n")

=== Doc 1 ===
Residential rental agreements - Before moving in - Report an issue with an advertised rental property - Report an issue with an advertised rental property Prospective renters with concerns about an advertised rental property can report an issue to us. Find out more, go to .
{'h3': 'Report an issue with an advertised rental property', 'url': 'the_url'}


=== Doc 2 ===
Paying for utilities and services - Very high usage charges - If a renter gets a very high bill for using water, gas or electricity because there is a hidden fault at the property, the rental provider might have to pay the excess amount. These very high bills are also called excessive usage charges. The rental provider only has to pay excessive usage charges if: the renter notified the rental provider, as soon as possible, of the excessive charges and the fault that caused them the renter didn’t cause the fault, and the service provider isn’t responsible for the fault. The rental provider must also reimburse 

We can see that as the complexity of the query increases, the recall rate decreases.